In [16]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))  # make src/ importable
 
import torch
from src.dataset import load_cached_paths
from src.infill_generator import infill_and_export, create_test_partial_xml, load_partial_xml
from src.tokenizer import identify_empty_measures

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Test infill gap detection

In [17]:
from pathlib import Path

# Create a test partial XML file with empty measures
TEST_XML_PATH = "output/test_partial.xml"
EMPTY_MEASURES = [3, 4, 5]  # measures to leave empty (these will be filled)

Path("output").mkdir(exist_ok=True)
create_test_partial_xml(TEST_XML_PATH, empty_measures=EMPTY_MEASURES)
print(f"Created test file: {TEST_XML_PATH}")
print(f"Empty measures: {EMPTY_MEASURES}")

# Verify gaps are detected
score, gaps = load_partial_xml(TEST_XML_PATH)
print(f"Detected gaps: {gaps}")

Created test file: output/test_partial.xml
Empty measures: [3, 4, 5]
[<music21.stream.Part 0x1d4fad27890>]
{1, 2, 3, 4, 5, 6, 7, 8}
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
Detected gaps: [(3, 5)]


<hr>

### Generation Params

In [18]:
INFILL_TEMPERATURE = 0.8
INFILL_TOP_K = 50
INFILL_MAX_NEW = 500
FILL_MEASURES = [ 3, 4, 7, 8, 9 ]

### Test Generation on Dataset Score

In [19]:
import music21 as m21
from pathlib import Path

# Get a score from the dataset
paths = load_cached_paths()
if not paths:
    print("No dataset found. Run Download Dataset first.")
else:
    # Take the first score
    original_path = paths[0]
    print(f"Using score: {original_path}")
    
    # Parse the score
    original_score = m21.converter.parse(original_path)
    
    # Find how many measures it has
    if original_score.parts:
        measures = list(original_score.parts[0].getElementsByClass(m21.stream.Measure))
        n_measures = len(measures)
        print(f"Original score has {n_measures} measures")
        
        if n_measures >= 6:
            # Create a modified score with measures 3-4 emptied
            modified_score = m21.stream.Score()
            
            for part in original_score.parts:
                new_part = m21.stream.Part()
                for measure in part.getElementsByClass(m21.stream.Measure):
                    m_num = measure.number if measure.number is not None else 0
                    new_measure = m21.stream.Measure(number=m_num)
                    
                    if m_num in FILL_MEASURES:
                        rest = m21.note.Rest(quarterLength=4)
                        rest.style.hideObjectOnPrint = True
                        
                        new_measure.elements = []
                        new_measure.insert(0, rest)
                    else:
                        for el in measure:
                            if isinstance(el, (m21.note.Note, m21.note.Rest, m21.chord.Chord, m21.stream.Voice)):
                                new_measure.insert(el.offset, el)

                    new_part.append(new_measure)
                modified_score.append(new_part)
                        
            # Verify gaps detected
            gaps = identify_empty_measures(modified_score)
            print(f"Detected gaps: {gaps}")
            
            # Run infill
            print("\nRunning infill on real score...")
            print("=" * 50)
            result_paths = infill_and_export(
                score=modified_score,
                max_new=INFILL_MAX_NEW,
                temperature=INFILL_TEMPERATURE,
                top_k=INFILL_TOP_K,
                log=print,
                device=DEVICE,
            )
            print("=" * 50)
            
            # if result_paths["xml"]:
            #     print("\n--- Filled Score Preview ---")
            #     show_text_score(result_paths["xml"], print)
        else:
            print(f"Score too short ({n_measures} measures), need at least 6")

Using score: storage\data\openscore\scores_Abbott,_Jane_Bingham___Just_for_Today_lc6583477.mxl
Original score has 41 measures
[<music21.stream.Part 0x1d4fb101a90>, <music21.stream.Part 0x1d4fb174550>, <music21.stream.Part 0x1d4fad247d0>]
{1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41}
<music21.note.Rest dotted-half>, True, False
<music21.note.Rest dotted-half>, True, False
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest dotted-half>, True, False
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.Rest whole>, True, True
<music21.note.R